In [1]:
!pip install python-dotenv

In [21]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv
from datetime import datetime, timedelta
import time

# 加载 .env 文件中的变量
load_dotenv()

# 获取 API key
API_KEY = os.getenv("POLYGON_API_KEY")
SYMBOL = "BRK.B"  # 股票代码

# 创建数据存储文件夹（如果文件夹不存在）
folder = "data_polygon"
if not os.path.exists(folder):
    os.makedirs(folder)

# 获取今天的日期
end_date = datetime.today()
start_date = end_date - timedelta(days=365 * 2)  # 获取过去 2 年的数据

# 转换为 'YYYY-MM-DD' 格式
start_date_str = start_date.strftime('%Y-%m-%d')
end_date_str = end_date.strftime('%Y-%m-%d')

# 构建文件名
file_name = f"{SYMBOL}_{end_date_str}.csv"
file_path = os.path.join(folder, file_name)

# 检查文件是否已存在
if os.path.exists(file_path):
    print(f"文件 {file_name} 已存在，跳过 API 调用。")
else:
    # 文件不存在，调用 API 获取数据
    print(f"文件 {file_name} 不存在，开始调用 API。")
    
    # 构建请求 URL：一次性请求 过去 2 年的数据
    url = f"https://api.polygon.io/v2/aggs/ticker/{SYMBOL}/range/1/day/{start_date_str}/{end_date_str}?apiKey={API_KEY}"

    # 请求数据
    response = requests.get(url)
    data = response.json()

    # 存储所有数据的列表
    all_data = []

    if "results" in data:
        all_data.extend(data["results"])
        print(f"已获取数据: {start_date_str} 到 {end_date_str}")
    else:
        print(f"没有返回数据: {start_date_str} 到 {end_date_str}")
    
    # 为了避免超过 API 调用限制，暂停 12 秒
    time.sleep(12)  # 等待 12 秒，确保每分钟最多请求 5 次

    # 将所有数据保存为 DataFrame
    df = pd.DataFrame(all_data)

    # 打印获取到的 DataFrame
    print(f"总共获取了 {len(df)} 条数据")
    print(df)

    # 保存数据到 CSV
    df.to_csv(file_path, index=False)
    print(f"数据已保存到 {file_path}")


文件 BRK.B_2025-05-11.csv 不存在，开始调用 API。
已获取数据: 2023-05-12 到 2025-05-11
总共获取了 500 条数据
              v        vw        o       c         h         l              t  \
0     1938264.0  322.0491  323.820  322.49  324.2422  320.5400  1683864000000   
1     2191609.0  322.6977  322.890  323.53  323.8300  320.1300  1684123200000   
2     2139996.0  323.6697  322.460  323.75  324.6900  322.3550  1684209600000   
3     3047626.0  326.8602  325.020  327.39  328.2600  324.8200  1684296000000   
4     2808329.0  328.5298  326.870  329.76  329.9800  325.8500  1684382400000   
..          ...       ...      ...     ...       ...       ...            ...   
495  16380216.0  512.5374  520.080  512.15  521.1800  502.8001  1746417600000   
496   6088592.0  512.4108  509.570  512.33  515.7500  507.9900  1746504000000   
497   5581423.0  517.5826  515.015  518.22  520.2500  513.0000  1746590400000   
498   5018581.0  516.5777  520.980  513.25  521.2591  513.0401  1746676800000   
499   3621402.0  513.3999 